# 08 · Stage 6 final combination + ablation (evaluated on the held-out TEST set)


**Owner:** member A + D · **GPU:** ≈ 3–8 h depending on which components were kept · **Needs:** all decisions from 04–07

Only components that showed a meaningful gain (`ml.component_gain`: ≥ +1 mAP@0.5:0.95, or ≥ +1 APS without losing > 0.5 mAP) are combined.
Cumulative ablation (each row adds one thing; rows A–D already exist from earlier stages):

| Row | Model | Data | Res | Tiling | Init |
|---|---|---|---|---|---|
| A | YOLOv7 | 500 | 640 | – | COCO |
| B | YOLOv7 | 500 | best | – | COCO |
| C | best model | 500 | best | – | COCO |
| D | best model | 2000 | best | – | COCO |
| E | best model | 2000 | best/tile | if kept | COCO |
| F = final | best model | 2000 | best/tile | if kept | ground if kept |
| LOO | final − tiling (only when both components are kept) | | | | |

All rows are evaluated on **`aerial_test`** (never used before this notebook) as well as val. Components that were tested but not kept are
reported from their 1000-image experiments ("tested, not adopted").

In [2]:
# ===== 0. Setup — identical in every notebook. Runtime > Change runtime type > T4 GPU =====
SMOKE_TEST = False   # True: 16 train / 8 val images, 1 epoch — checks the whole pipeline in minutes
DRIVE_ROOT = "/content/drive/MyDrive/MAVREC"   # shared team folder (each member: "Add shortcut to Drive")

import os, sys, subprocess, importlib
try:
    from google.colab import drive
    try:
        os.listdir("/content/drive/MyDrive")   # raises if the Drive FUSE mount went stale
        drive.mount("/content/drive")
    except OSError:  # "Transport endpoint is not connected" after a Drive hiccup: remount
        drive.mount("/content/drive", force_remount=True)
except ImportError:
    print("Not in Colab — make sure DRIVE_ROOT exists locally.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.173", "transformers==5.18.0", "pycocotools>=2.0.8", "timm>=1.0.15", "tabulate", "tensorboard", "seaborn"], check=True)

CODE_DIR = os.path.join(DRIVE_ROOT, "Final_Project_Code")
assert os.path.exists(os.path.join(CODE_DIR, "mavrec_lib.py")), "Run 00_Setup_and_Data_Preparation first (it installs mavrec_lib.py)."
sys.path.insert(0, CODE_DIR)
import mavrec_lib as ml; importlib.reload(ml)
CFG = ml.setup(DRIVE_ROOT, smoke=SMOKE_TEST or None)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
N_SMALL, N_SCREEN, N_LARGE = ml.TRAIN_SIZES      # 500, 1000, 2000

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment:
  python       : 3.13.15
  torch        : 2.11.0+cu130
  cuda         : True
  torchvision  : 0.26.0+cu130
  ultralytics  : 8.4.173
  transformers : 5.18.0
  pycocotools  : 2.0.11
  numpy        : 2.1.3
  lib          : 1.0.0
  gpu          : Tesla T4
  gpu_mem_gb   : 14.6


In [3]:
BEST_MODEL = ml.get_decision("best_model", required=True)
BEST_RES = ml.get_decision("best_res", required=True)
USE_TILING = bool(ml.get_decision("use_tiling", False))
USE_GROUND = bool(ml.get_decision("use_ground", False))
TILING_MODE = ml.get_decision("tiling_mode", "sliced")
G_ID = ml.get_decision("ground_exp", None)
E = ml.DEFAULT_EPOCHS[BEST_MODEL]
print(dict(BEST_MODEL=BEST_MODEL, BEST_RES=BEST_RES, USE_TILING=USE_TILING, USE_GROUND=USE_GROUND, TILING_MODE=TILING_MODE))

{'BEST_MODEL': 'yolov8m', 'BEST_RES': 1280, 'USE_TILING': False, 'USE_GROUND': False, 'TILING_MODE': 'sliced'}


## Build the rows (re-uses finished experiments; trains only what is missing)

In [4]:
ROWS = []   # (row label, experiment_id, inference mode)
def add(label, spec, mode=None, modes=None):
    ml.run_experiment(spec, eval_splits=("val", "test"), eval_modes=modes or [mode])
    ROWS.append((label, spec["experiment_id"], mode or ("sliced" if spec["tiling"] == "tile_train" else "full")))

add("A  YOLOv7 · 500 · 640 (Review 2 setting, re-run)", ml.make_spec("yolov7", f"aerial_train_{N_SMALL}", 640, stage="S0"))
add(f"B  + resolution {BEST_RES}", ml.make_spec("yolov7", f"aerial_train_{N_SMALL}", BEST_RES, stage="S0" if BEST_RES in (640, 960) else "S6"))
add(f"C  + architecture ({BEST_MODEL})", ml.make_spec(BEST_MODEL, f"aerial_train_{N_SMALL}", BEST_RES, stage="S3"))
add(f"D  + data ({N_LARGE} frames)", ml.make_spec(BEST_MODEL, f"aerial_train_{N_LARGE}", BEST_RES, stage="S3"))
tile_kw = dict(tiling="tile_train", epochs=max(3, round(E / 5)), full_res=1280)
if USE_TILING:
    add("E  + tiling", ml.make_spec(BEST_MODEL, f"aerial_train_{N_LARGE}", stage="S6", **tile_kw), mode=TILING_MODE, modes=[TILING_MODE])
if USE_GROUND:
    assert G_ID, "ground_exp decision missing (notebook 07)"
    kw = dict(init=f"exp:{G_ID}", pretraining="COCO>MAVREC-ground")
    if USE_TILING:
        add("F  + ground pretraining (FINAL)", ml.make_spec(BEST_MODEL, f"aerial_train_{N_LARGE}", stage="S6", **tile_kw, **kw), mode=TILING_MODE, modes=[TILING_MODE])
        add("LOO final − tiling", ml.make_spec(BEST_MODEL, f"aerial_train_{N_LARGE}", BEST_RES, stage="S6", **kw))
    else:
        add("F  + ground pretraining (FINAL)", ml.make_spec(BEST_MODEL, f"aerial_train_{N_LARGE}", BEST_RES, stage="S6", **kw))
FINAL_ID = ROWS[-1][1] if not ROWS[-1][0].startswith("LOO") else ROWS[-2][1]
FINAL_MODE = [m for l, e, m in ROWS if e == FINAL_ID][0]
ml.set_decision("final_model", FINAL_ID, "last cumulative ablation row")
ml.set_decision("final_mode", FINAL_MODE, "inference mode of the final model")
ROWS

[S0_yolov7_n500_r640_full_COCO_s42] done — skipping (force=True to redo)
Pack 'aerial_1280' unpacked to local disk in 53s
[S6_yolov7_n500_r1280_full_COCO_s42] training yolov7 | data=aerial_train_500 | res=1280 | epochs=40 | batch=2 | init=coco
/usr/bin/python3 train.py --resume /content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S6_yolov7_n500_r1280_full_COCO_s42/train/weights/last.pt
Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
IDetect.fuse


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  predicted 538 images (full) in 33s, 148504 detections
[S6_yolov7_n500_r1280_full_COCO_s42] val|full|r1280: mAP50-95=37.572 AP50=65.914 APS=27.346 APt=10.073 R=82.789 FPS=24.9
Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
IDetect.fuse
  predicted 1614 images (full) in 96s, 433316 detections
[S6_yolov7_n500_r1280_full_COCO_s42] test|full|r1280: mAP50-95=35.473 AP50=63.145 APS=25.706 APt=12.793 R=80.601 FPS=24.34
[S3_yolov8m_n500_r1280_full_COCO_s42] already trained -> /content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S3_yolov8m_n500_r1280_full_COCO_s42/train/weights/last.pt
[S3_yolov8m_n500_r1280_full_COCO_s42] val|full|r1280 already evaluated
  predicted 1614 images (full) in 96s, 266212 detections
[S3_yolov8m_n500_r1280_full_COCO_s42] test|full|r1280: mAP50-95=59.246 AP50=87.717 APS=34.405 APt=16.227 R=84.536 FPS=30.05
[S3_yolov8m_n2000_r1280_full_COCO_s42] already trained -> /content/drive/MyDrive/MAVREC/Final_Project_Results/

[('A  YOLOv7 · 500 · 640 (Review 2 setting, re-run)',
  'S0_yolov7_n500_r640_full_COCO_s42',
  'full'),
 ('B  + resolution 1280', 'S6_yolov7_n500_r1280_full_COCO_s42', 'full'),
 ('C  + architecture (yolov8m)',
  'S3_yolov8m_n500_r1280_full_COCO_s42',
  'full'),
 ('D  + data (2000 frames)', 'S3_yolov8m_n2000_r1280_full_COCO_s42', 'full')]

## Ablation table (val and test)

In [5]:
df = ml.results_df()
recs = []
for label, eid, mode in ROWS:
    for split in ("val", "test"):
        r = df[(df.experiment_id == eid) & (df.eval_split == split) & (df.inference_mode == mode)]
        if len(r):
            recs.append(dict(row=label, split=split, **r.iloc[0][["experiment_id", "mAP50_95", "mAP50", "APS", "APt", "APM", "recall", "F1", "FPS", "latency_ms", "parameters_M", "train_time_h"]].to_dict()))
ABL = pd.DataFrame(recs)
for split in ("val", "test"):
    t = ABL[ABL.split == split].copy()
    t["Δ mAP vs prev"] = t.mAP50_95.diff().round(2); t["Δ APS vs prev"] = t.APS.diff().round(2)
    print(split.upper()); display(t.drop(columns=["split"]))
    ml.save_table(t, f"S6_ablation_{split}")

VAL


,row,experiment_id,mAP50_95,mAP50,APS,...,latency_ms,parameters_M,train_time_h,Δ mAP vs prev,Δ APS vs prev
0,"A YOLOv7 · 500 · 640 (Review 2 setting, re-run)",S0_yolov7_n500_r640_full_COCO_s42,23.853,44.305,7.298,...,20.51,36.53,0.775,NaN,NaN
2,B + resolution 1280,S6_yolov7_n500_r1280_full_COCO_s42,37.572,65.914,27.346,...,40.15,36.53,0.543,13.72,20.05
4,C + architecture (yolov8m),S3_yolov8m_n500_r1280_full_COCO_s42,63.086,89.694,34.456,...,39.01,25.86,0.716,25.51,7.11
6,D + data (2000 frames),S3_yolov8m_n2000_r1280_full_COCO_s42,70.811,92.674,43.506,...,32.96,25.86,0.739,7.73,9.05


saved /content/drive/MyDrive/MAVREC/Final_Project_Results/tables/S6_ablation_val.csv
TEST


,row,experiment_id,mAP50_95,mAP50,APS,...,latency_ms,parameters_M,train_time_h,Δ mAP vs prev,Δ APS vs prev
1,"A YOLOv7 · 500 · 640 (Review 2 setting, re-run)",S0_yolov7_n500_r640_full_COCO_s42,21.870,41.381,6.801,...,21.55,36.53,0.775,NaN,NaN
3,B + resolution 1280,S6_yolov7_n500_r1280_full_COCO_s42,35.473,63.145,25.706,...,41.09,36.53,0.543,13.60,18.90
5,C + architecture (yolov8m),S3_yolov8m_n500_r1280_full_COCO_s42,59.246,87.717,34.405,...,33.28,25.86,0.716,23.77,8.70
7,D + data (2000 frames),S3_yolov8m_n2000_r1280_full_COCO_s42,68.554,91.165,43.848,...,33.32,25.86,0.739,9.31,9.44


saved /content/drive/MyDrive/MAVREC/Final_Project_Results/tables/S6_ablation_test.csv


In [6]:
t = ABL[(ABL.split == "test") & ~ABL.row.str.startswith("LOO")]
fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
for a, met in zip(ax, ["mAP50_95", "APS"]):
    v = t[met].values
    a.bar(range(len(v)), np.diff(np.r_[0, v]), bottom=np.r_[0, v[:-1]], color=["#888"] + ["#2a9d8f" if d >= 0 else "#e76f51" for d in np.diff(v)])
    for i, x in enumerate(v): a.text(i, x + 0.3, f"{x:.1f}", ha="center", fontsize=9)
    a.set_xticks(range(len(v))); a.set_xticklabels([r.split("  ")[0] for r in t.row]); a.set_title(f"Cumulative ablation — {met} (test)")
plt.tight_layout(); ml.savefig("S6_ablation_waterfall.png"); plt.show()

saved /content/drive/MyDrive/MAVREC/Final_Project_Results/figures/S6_ablation_waterfall.png


## Components tested but not adopted (from the 1000-frame studies)

In [7]:
df = ml.results_df()
NOT = df[df.stage.isin(["S4", "S5"]) & (df.eval_split == "val")]
display(NOT[["experiment_id","stage","model_version","dataset_size","resolution","tiling","inference_mode","pretraining","mAP50_95","mAP50","APS","APt","APM","recall","F1","FPS","GPU_memory_infer_GB","parameters_M","train_time_h"]])

,experiment_id,stage,model_version,dataset_size,resolution,...,F1,FPS,GPU_memory_infer_GB,parameters_M,train_time_h
24,S4_yolov8m_n1000_r640_tile_COCO_s42,S4,yolov8m,1000,640.0,...,92.936,2.96,0.32,25.86,0.949
25,S4_yolov8m_n1000_r640_tile_COCO_s42,S4,yolov8m,1000,640.0,...,91.125,2.45,0.32,25.86,0.949
26,S5_yolov8m_g2000_r1280_full_COCO_s42_groundonly,S5,yolov8m,2000,1280.0,...,18.481,29.01,0.15,25.86,0.225
27,S5_yolov8m_n1000_r1280_full_COCO2MAVRECground_s42,S5,yolov8m,1000,1280.0,...,90.828,33.18,0.15,25.86,0.774
28,S5_yolov8m_n1000_r1280_full_COCO_s42_longer,S5,yolov8m,1000,1280.0,...,92.804,30.84,0.15,25.86,1.760


## Per-class AP and qualitative comparison: baseline A vs final

In [8]:
A_ID = ROWS[0][1]
pa = df[(df.experiment_id == A_ID) & (df.eval_split == "test")].iloc[0]
pf = df[(df.experiment_id == FINAL_ID) & (df.eval_split == "test") & (df.inference_mode == FINAL_MODE)].iloc[0]
PC = pd.DataFrame({"A (YOLOv7 500/640)": pa.per_class_AP, "Final": pf.per_class_AP}).reindex(ml.CLASS_NAMES)
PC["test GT boxes"] = pd.Series(ml.dataset_stats(ml.load_dataset("aerial_test"))["class_counts"]).reindex(ml.CLASS_NAMES)
display(PC.round(1)); ml.save_table(PC.reset_index().rename(columns={"index": "class"}), "S6_per_class_AP_test")
ml.prediction_grid([("A: YOLOv7 500/640", pa.preds_path, pa.conf_at_best_f1), ("Final", pf.preds_path, pf.conf_at_best_f1)],
                   "aerial_test", "aerial_1280", n=3, out_path=os.path.join(CFG["figures_dir"], "S6_qualitative_test.png"))

,A (YOLOv7 500/640),Final,test GT boxes
tram,0.0,85.8,12
bicycle,7.8,41.1,4691
van,9.0,77.9,1682
truck,63.1,84.4,258
bus,0.0,90.2,22
person,12.1,43.6,9070
car,32.9,59.4,56086
other,26.8,58.1,21937
streetlight,39.7,72.4,20194
traffic_light,27.4,72.6,6565


saved /content/drive/MyDrive/MAVREC/Final_Project_Results/tables/S6_per_class_AP_test.csv
